In [11]:
#%pip install pandas matplotlib seaborn sqlalchemy ipython-sql==0.4.1 prettytable==2.5.0
#%pip install tabulate  
#%pip install plotly.express

In [ ]:
%pip install plotly

In [20]:
%pip install nbformat>=4.2.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# VDP Analysis
This notebook will be working with the 'VDP_dataset.csv' file, which was extracted from the Dune query at dune.com/queries/7717567. The table will have the following Columns:
 
| Column | Description |
|---|---|
| id | |
| name | |
| website | |
| auth_address | |
| current_commission | |
| total_stake | |
| vdp_stake | |
| Organic_stake | |
| initial_tier | |
| current_tier | |
| dependency_ratio | |
| Rewards_USD | |
| MonthlyReturns_USD | |
| EstimatedMonthlyReturns_USD | |
| underwater_status | |
| graduation_status | |

In [3]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///vdpdata.db')

csv_file = 'Vdp_dataset.csv'
table_name = 'vdp'
df = pd.read_csv(csv_file)
df.to_sql(table_name, engine, if_exists='replace', index=False)
print('successful')

successful


In [4]:
import matplotlib.pyplot as plt
import pandas as pd

query = """SELECT *
FROM 'vdp'
LIMIT 3"""

df = pd.read_sql(query, engine)
df

,id,name,website,auth_address,current_commission,total_stake,vdp_stake,organic_stake,initial_tier,current_tier,dependency_ratio,Rewards_USD,MonthlyReturns_USD,EstimatedMonthlyReturns_USD,underwater_status,graduation_status
0,1,Monad Foundation - mf-mainnet-val-tsw-fra-001,https://www.monad.foundation/,0x675130B3CBB3E5575959AEC01978390B5EC4971A,0,1.000617e+07,0.0,1.000617e+07,NaN,NaN,0.000000,0.000,0.00,0.000,1,0
1,2,Monad Foundation - mf-mainnet-val-lsn-jfk-011,https://www.monad.foundation/,0xFB1915F56855DEF26422FF3C600A2EAD57830521,0,1.001232e+07,0.0,1.001232e+07,NaN,NaN,0.000000,0.000,0.00,0.000,1,0
2,3,gmonads.com,https://www.gmonads.com/,0xA63DD8FC7303BDD6CD66A66A49D4171146F89809,15,2.597674e+08,80000000.0,1.797674e+08,Tier 2a,Tier 2a,0.307968,36002.155,5412.56,14511.323,0,1


- How many validators received a delegation?
- Which delegation tier did each validator receive? 
- How concentrated is VDP stake among validators?


In [5]:
query = """SELECT COUNT(DISTINCT id) as "Total VDP Validators"
FROM 'vdp'
WHERE initial_tier IS NOT NULL"""
df = pd.read_sql(query,engine)
df

,Total VDP Validators
0,206


In [6]:
query = """SELECT id, name, initial_tier
FROM 'vdp'
WHERE initial_tier IS NOT NULL"""
df = pd.read_sql(query, engine)
df

,id,name,initial_tier
0,3,gmonads.com,Tier 2a
1,4,B-Harvest,Tier 2a
2,5,Alchemy,Tier 2a
3,6,ProStaking,Tier 2a
4,7,Staking4All,Tier 2a
...,...,...,...
201,214,Lunar Strategy,Tier 2b
202,215,Pointgroup,Tier 3
203,216,Astrosynx,Tier 3
204,217,InfraSingularity,Tier 3


In [7]:
import plotly.express as px

query = """SELECT initial_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)
label = df["initial_tier"]
size = df["total_delegators"]
fig = px.pie(df, values=size, names=label, title="VDP Distribution Chart", hole=0.0)
fig.show()

In [21]:
# How Concentrated is VDP Stake

query="""SELECT SUM(vdp_stake) * 1.0 / SUM(total_stake) * 1.0 AS concentration_ratio
FROM 'vdp'
"""
df = pd.read_sql(query,engine)
df

,concentration_ratio
0,0.738648


Herfindal-Hirschman Index (HHI)

In [22]:
query ="""with shares AS (SELECT vdp_stake * 1.0 / SUM(vdp_stake) OVER() AS share
FROM vdp)
select sum(power(share, 2)) * 10000 AS hhi_10000
FROM shares"""
df = pd.read_sql(query, engine)
df

,hhi_10000
0,54.390096
